In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [2]:
db_path = Path("databases") / "url_features_extracted1.csv"
dataset = pd.read_csv(db_path)

# Display the number of rows and columns in the dataset
print("Nombre de lignes et de colonnes du dataset : ", dataset.shape)

# Display the dataset features (column names)
print("Features du dataset : ", dataset.columns)

Nombre de lignes et de colonnes du dataset :  (101219, 18)
Features du dataset :  Index(['URL', 'url_length', 'has_ip_address', 'dot_count', 'https_flag',
       'url_entropy', 'token_count', 'subdomain_count', 'query_param_count',
       'tld_length', 'path_length', 'has_hyphen_in_domain', 'number_of_digits',
       'tld_popularity', 'suspicious_file_extension', 'domain_name_length',
       'percentage_numeric_chars', 'ClassLabel'],
      dtype='object')


In [3]:
# Check whether the dataset contains any NaN values
print("Nombre de valeurs NaN dans le dataset : ", dataset.isnull().sum().sum())

# Remove rows containing NaN values
dataset = dataset.dropna()

# Check whether the dataset contains any NaN values
print("Nombre de valeurs NaN dans le dataset : ", dataset.isnull().sum().sum())

Nombre de valeurs NaN dans le dataset :  1
Nombre de valeurs NaN dans le dataset :  0


In [4]:
# Randomly split the dataset: 80% for training and 20% for testing
from sklearn.model_selection import train_test_split

X = dataset.drop(columns=["URL", "ClassLabel"])
y = dataset["ClassLabel"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(y_test)

64156    1.0
22321    0.0
12633    0.0
62138    0.0
65468    1.0
        ... 
3147     0.0
18496    0.0
32663    0.0
93067    1.0
15911    0.0
Name: ClassLabel, Length: 20244, dtype: float64


1. Random forest

In [5]:
# Train a classification model with Random Forest, optimized using GridSearchCV
from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

my_kfold = KFold(n_splits=10, shuffle=True, random_state=42)

random_forest_grid = GridSearchCV(estimator=RandomForestClassifier(random_state=42), 
                                  param_grid=param_grid, cv=my_kfold, n_jobs=-1)

random_forest_grid.fit(X_train, y_train)

# The best model
print(random_forest_grid.best_params_)


{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


{'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100}


In [7]:
from sklearn.metrics import f1_score

# Train using the best model

best_random_forest = random_forest_grid.best_estimator_
best_random_forest.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_random_forest.predict(X_test)

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix and F1 score
print("Matrice de confusion:")
print(cm)
print("F1-score:")
print(f1_score(y_test, y_pred))



Matrice de confusion:
[[12748     1]
 [    8  7487]]
F1-score:
0.9993993192284589


In [9]:
# Check the most important features in the Random Forest model
importances = best_random_forest.feature_importances_
print("Importance des features : ", importances)


Importance des features :  [1.66188840e-02 1.24515209e-02 4.33782317e-02 1.28196854e-01
 4.32789156e-03 1.40742607e-01 2.24558634e-02 7.25157145e-05
 1.40724962e-02 4.05047598e-01 1.93129299e-04 9.99841245e-02
 1.45052569e-03 1.18073507e-03 1.22064585e-02 9.76205646e-02]


2. Gradient boosting

In [10]:
# Train a classification model with XGBoost, optimized using GridSearchCV

from xgboost import XGBClassifier

param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

XG_boost_grid = GridSearchCV(estimator=XGBClassifier(random_state=42),
                             param_grid=param_grid_xgb, cv=my_kfold, n_jobs=-1)

XG_boost_grid.fit(X_train, y_train)

# The best model
print(XG_boost_grid.best_params_)





{'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 100}


In [11]:
# Train using the best model

best_xgb = XG_boost_grid.best_estimator_
best_xgb.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_xgb.predict(X_test)

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix and F1 score
print("Matrice de confusion:")
print(cm)
print("F1-score:")
print(f1_score(y_test, y_pred))



Matrice de confusion:
[[12746     3]
 [    7  7488]]
F1-score:
0.9993327105298279


In [12]:
# Check the most important features in the XGBoost model
feature_importances = best_xgb.feature_importances_
print("Importance des features : ", feature_importances)

Importance des features :  [2.8757600e-03 0.0000000e+00 2.1976072e-02 4.3673091e-02 8.8788039e-04
 6.4877183e-03 1.9052295e-02 0.0000000e+00 6.6955882e-04 8.7796539e-01
 2.0698451e-03 2.0960604e-03 9.2607019e-03 1.0104178e-02 1.1194141e-03
 1.7619922e-03]


3. Logistic regression

In [14]:
from sklearn.linear_model import LogisticRegression

# Train a classification model with Logistic Regression, optimized using GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

my_kfold = KFold(n_splits=10, shuffle=True, random_state=42)

logistic_regression_grid = GridSearchCV(estimator=LogisticRegression(random_state=42), 
                                       param_grid=param_grid, cv=my_kfold, n_jobs=-1)

logistic_regression_grid.fit(X_train, y_train)

# The best model
print(logistic_regression_grid.best_params_)


{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}


In [15]:
# Train using the best model

best_logistic = logistic_regression_grid.best_estimator_
best_logistic.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_logistic.predict(X_test)

# Calculate the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display the confusion matrix and F1 score
print("Matrice de confusion:")
print(cm)
print("F1-score:")
print(f1_score(y_test, y_pred))



Matrice de confusion:
[[12694    55]
 [   22  7473]]
F1-score:
0.9948745257272182


In [16]:
# Check the most important features in the Logistic Regression model
feature_importances = best_logistic.coef_[0]
print("Importance des features : ", feature_importances)

Importance des features :  [-0.26560922 -5.67044139 -4.76585118  9.08834286  1.26011824 -2.13915205
  8.82279342  3.07774434 -0.22955541  0.32451972 -0.52376059  0.13072476
  1.78040889 -1.63628649  0.10147627 -0.14532799]


4. Comparison of the three model families

In [ ]:
# Compare the F1 scores of the three models (Random Forest, XGBoost, and Logistic Regression)

